# Test dei metodi per l'allineamento

Questo notebook ha l'obiettivo di provare i vari metodi di allineamento (ECC, SIFT, ORB) con diversi tipi di immagini e visualizzarne i risultati.

# Definizioni e caricamenti

### Caricamento dei moduli

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Funzioni di aiuto matplotlib

In [ ]:
STD_FIGSIZE = (20, 20)

def show_image(image, title, cmap=None, figsize=STD_FIGSIZE):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    figure = plt.figure(figsize=figsize)
    subplot = figure.add_subplot(1, 1, 1)
    subplot.set_title(title)
    subplot.axis('off')
    subplot.imshow(image, cmap=cmap)

def show_side_images(image_1, title_1, image_2, title_2, cmap=None, figsize=STD_FIGSIZE):
    if len(image_1.shape) == 3:
        image_1 = cv2.cvtColor(image_1, cv2.COLOR_BGR2RGB)
    if len(image_2.shape) == 3:
        image_2 = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
    figure = plt.figure(figsize=figsize)
    subplots = figure.subplots(1, 2)
    figure.subplots_adjust(wspace=0.01)
    subplots[0].set_title(title_1)
    subplots[0].axis('off')
    subplots[0].imshow(image_1, cmap=cmap)
    subplots[1].set_title(title_2)
    subplots[1].axis('off')
    subplots[1].imshow(image_2, cmap=cmap)

### Funzioni di aiuto OpenCV

#### Funzioni semplici

In [ ]:
def apply_clahe(image, clipLimit=2.0, tileGridSize=(8,8)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    output_image = clahe.apply(image)
    return output_image

def blur_image(image, strength=5):
    return cv2.GaussianBlur(image, (strength, strength), 0)

#### Filtri

In [ ]:
from cv2 import DMatch

DEF_FILTER_DISTANCE = 20
DEF_LOWES_RATIO = 0.75

def filter_matches_with_lowes_ratio(matches: list[DMatch], threshold=DEF_LOWES_RATIO):
    filtered_matches = []
    for m in matches:
        if len(m) == 2 and m[0].distance < threshold * m[1].distance:
            filtered_matches.append(m[0])
    return filtered_matches

def filter_matches_by_euclidean_distance(matches: list[DMatch], keypoints: tuple[list, list], distance=DEF_FILTER_DISTANCE):
    filtered_matches = []
    for m in matches:
        euclidean_distance = np.linalg.norm(np.array(keypoints[0][m.queryIdx].pt) - np.array(keypoints[1][m.trainIdx].pt))
        if euclidean_distance < distance:
            filtered_matches.append(m)
    return filtered_matches

#### Funzioni di match

In [ ]:
from cv2 import DMatch

DEF_NFEATURES = 10000
DEF_SIFT_NFEATURES = DEF_NFEATURES
DEF_ORB_NFEATURES = DEF_NFEATURES

def compute_sift_matches(image_1, image_2, nfeatures=DEF_SIFT_NFEATURES, use_lowes_ratio=False):
    sift = cv2.SIFT_create(nfeatures=nfeatures)
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=not use_lowes_ratio)

    # Rileva i punti chiave e calcola i descrittori
    keypoints_1, descriptors_1 = sift.detectAndCompute(image_1, None)
    keypoints_2, descriptors_2 = sift.detectAndCompute(image_2, None)

    # Corrispondenze tra i descrittori
    if use_lowes_ratio:
        matches = bf.knnMatch(descriptors_1, descriptors_2, k=2)
        matches = filter_matches_with_lowes_ratio(matches)
    else:
        matches = bf.match(descriptors_1, descriptors_2)
    matches = sorted(matches, key=lambda x: x.distance)
    return matches, (keypoints_1, keypoints_2)

def compute_orb_matches(image_1, image_2, nfeatures=DEF_ORB_NFEATURES, use_lowes_ratio=False):
    orb = cv2.ORB_create(nfeatures=nfeatures)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=not use_lowes_ratio)

    # Rileva i punti chiave e calcola i descrittori
    keypoints_1, descriptors_1 = orb.detectAndCompute(image_1, None)
    keypoints_2, descriptors_2 = orb.detectAndCompute(image_2, None)

    # Corrispondenze tra i descrittori
    if use_lowes_ratio:
        matches = bf.knnMatch(descriptors_1, descriptors_2, k=2)
        matches = filter_matches_with_lowes_ratio(matches)
    else:
        matches = bf.match(descriptors_1, descriptors_2)
    matches = sorted(matches, key=lambda x: x.distance)
    return matches, (keypoints_1, keypoints_2)

def extract_points(matches: list[DMatch], keypoints: tuple[list, list]):
    points = ([], [])
    for m in matches:
        points[0].append(keypoints[0][m.queryIdx].pt)
        points[1].append(keypoints[1][m.trainIdx].pt)
    return points

def compute_ransac_transform(points_1, points_2):
    points_1 = np.float32(points_1).reshape(-1,1,2)
    points_2 = np.float32(points_2).reshape(-1,1,2)
    H, mask = cv2.findHomography(points_2, points_1, cv2.RANSAC, 10.0)
    return H

def apply_ransac_transform(image, warp_matrix, shape=None):
    if shape is None:
        h, w = image.shape[:2]
    else:
        h, w = shape[:2]
    output_image = cv2.warpPerspective(image, warp_matrix, (w, h))
    return output_image

def compute_ecc_transform(image_1, image_2):
    # Inizializza i parametri per l'algoritmo ECC
    warp_mode = cv2.MOTION_TRANSLATION
    warp_matrix = np.eye(2, 3, dtype=np.float32)

    # Imposta i parametri di terminazione
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 5000, 1e-10)

    # Esegue l'algoritmo ECC
    cc, warp_matrix = cv2.findTransformECC(image_1, image_2, warp_matrix, warp_mode, criteria)
    return warp_matrix

def apply_ecc_transform(image, warp_matrix, shape=None):
    if shape is None:
        h, w = image.shape[:2]
    else:
        h, w = shape[:2]
    output_image = cv2.warpAffine(image, warp_matrix, (w, h), flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
    return output_image

### Apertura delle immagini

In [ ]:
label_free_path = "../../Materiale/Immagini/non_colorato.png"
stained_path = "../../Materiale/Immagini/colorato.png"

label_free = cv2.imread(label_free_path)
stained = cv2.imread(stained_path)

label_free_gray = cv2.cvtColor(label_free, cv2.COLOR_BGR2GRAY)
stained_gray = cv2.cvtColor(stained, cv2.COLOR_BGR2GRAY)

label_free_equalized = cv2.equalizeHist(label_free_gray)
stained_equalized = cv2.equalizeHist(stained_gray)

label_free_clahe = apply_clahe(label_free_gray, clipLimit=18.0, tileGridSize=(8, 8))
stained_clahe = apply_clahe(stained_gray, clipLimit=18.0, tileGridSize=(8, 8))

error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


# Visualizzazione delle immagini iniziali

In [ ]:
# show_side_images(label_free, "Non colorata", stained, "Colorata")
# show_side_images(label_free_gray, "Non colorata grigio", stained_gray, "Colorata grigio", cmap='gray')
# show_side_images(label_free_equalized, "Non colorata equalizzata", stained_equalized, "Colorata equalizzata", cmap='gray')
# show_side_images(label_free_clahe, "Non colorata CLAHE", stained_clahe, "Colorata CLAHE", cmap='gray')

figure, subplots = plt.subplots(4, 2, figsize=(10, 15))
figure.subplots_adjust(wspace=10, hspace=10)

subplots[0, 0].imshow(label_free)
subplots[0, 0].set_title("Label Free Originale")
subplots[0, 0].axis("off")
subplots[1, 0].imshow(label_free_gray, cmap='gray')
subplots[1, 0].set_title("Label Free Grayscale")
subplots[1, 0].axis("off")
subplots[2, 0].imshow(label_free_clahe, cmap='gray')
subplots[2, 0].set_title("Label Free CLAHE")
subplots[2, 0].axis("off")
subplots[3, 0].imshow(label_free_equalized, cmap='gray')
subplots[3, 0].set_title("Label Free equalizeHist")
subplots[3, 0].axis("off")

subplots[0, 1].imshow(stained)
subplots[0, 1].set_title("Stained Originale")
subplots[0, 1].axis("off")
subplots[1, 1].imshow(stained_gray, cmap='gray')
subplots[1, 1].set_title("Stained Grayscale")
subplots[1, 1].axis("off")
subplots[2, 1].imshow(stained_clahe, cmap='gray')
subplots[2, 1].set_title("Stained CLAHE")
subplots[2, 1].axis("off")
subplots[3, 1].imshow(stained_equalized, cmap='gray')
subplots[3, 1].set_title("Stained equalizeHist")
subplots[3, 1].axis("off")

plt.tight_layout()
plt.show()

# ECC

In [ ]:
warp_matrix = compute_ecc_transform(label_free_equalized, stained_equalized)
aligned_stained = apply_ecc_transform(stained, warp_matrix, shape=label_free.shape)

aligned_stained_gray = cv2.cvtColor(aligned_stained, cv2.COLOR_BGR2GRAY)
aligned_stained_equalized = cv2.equalizeHist(aligned_stained_gray)

difference_og = cv2.absdiff(label_free_equalized, stained_equalized)
difference_aligned = cv2.absdiff(label_free_equalized, aligned_stained_equalized)

show_side_images(label_free, "Non colorata", aligned_stained, "Colorata allineata")
show_side_images(difference_og, "Differenza assoluta tra immagini originali", difference_aligned, "Differenza assoluta tra immagini allineate", cmap='gray')

# SIFT

## Allinamento tramite Feature Matching con SIFT e immagini originali

In [ ]:
matches, keypoints = compute_sift_matches(label_free, stained)

### Filtraggio dei match

In [ ]:
filtered_matches = filter_matches_by_euclidean_distance(matches, keypoints)

### Display dei match

In [ ]:
drawn_matches = cv2.drawMatches(
    label_free, keypoints[0],
    stained, keypoints[1],
    filtered_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    matchesThickness=5
)
print(f"Matches: {len(filtered_matches)}")
show_image(drawn_matches, "Corrispondenze SIFT")

### Allineamento dell'immagine con RANSAC

In [ ]:
points = extract_points(filtered_matches, keypoints)
warp_matrix = compute_ransac_transform(points[0], points[1])
aligned_stained = apply_ransac_transform(stained, warp_matrix, shape=label_free.shape)
aligned_stained_gray = cv2.cvtColor(aligned_stained, cv2.COLOR_BGR2GRAY)
aligned_stained_equalized = cv2.equalizeHist(aligned_stained_gray)
aligned_stained_clahe = apply_clahe(aligned_stained_gray)

difference = cv2.absdiff(label_free_equalized, aligned_stained_equalized)

show_side_images(label_free, "Non colorata", aligned_stained, "Colorata allineata")
show_side_images(label_free_gray, "Non colorata", aligned_stained_gray, "Colorata allineata", cmap='gray')
show_side_images(label_free_equalized, "Non colorata", aligned_stained_equalized, "Colorata allineata", cmap='gray')
show_side_images(label_free_clahe, "Non colorata", aligned_stained_clahe, "Colorata allineata", cmap='gray')
show_image(difference, "Differenza assoluta tra immagini allineate", cmap='gray')

## Allinamento tramite Feature Matching con SIFT e immagini monocromatiche non equalizzate

In [ ]:
matches, keypoints = compute_sift_matches(label_free_gray, stained_gray)

### Filtraggio dei match

In [ ]:
filtered_matches = filter_matches_by_euclidean_distance(matches, keypoints)

### Display dei match

In [ ]:
drawn_matches = cv2.drawMatches(
    label_free, keypoints[0],
    stained, keypoints[1],
    filtered_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    matchesThickness=5
)
print(f"Matches: {len(filtered_matches)}")
show_image(drawn_matches, "Corrispondenze SIFT")

### Allineamento dell'immagine con RANSAC

In [ ]:
points = extract_points(filtered_matches, keypoints)
warp_matrix = compute_ransac_transform(points[0], points[1])
aligned_stained = apply_ransac_transform(stained, warp_matrix, shape=label_free.shape)
aligned_stained_gray = cv2.cvtColor(aligned_stained, cv2.COLOR_BGR2GRAY)
aligned_stained_equalized = cv2.equalizeHist(aligned_stained_gray)
aligned_stained_clahe = apply_clahe(aligned_stained_gray)

difference = cv2.absdiff(label_free_equalized, aligned_stained_equalized)

show_side_images(label_free, "Non colorata", aligned_stained, "Colorata allineata")
show_side_images(label_free_gray, "Non colorata", aligned_stained_gray, "Colorata allineata", cmap='gray')
show_side_images(label_free_equalized, "Non colorata", aligned_stained_equalized, "Colorata allineata", cmap='gray')
show_side_images(label_free_clahe, "Non colorata", aligned_stained_clahe, "Colorata allineata", cmap='gray')
show_image(difference, "Differenza assoluta tra immagini allineate", cmap='gray')

## Allinamento tramite Feature Matching con SIFT e immagini monocromatiche equalizzate

In [ ]:
matches, keypoints = compute_sift_matches(label_free_equalized, stained_equalized)
print(f"SIFT matches with equalizeHist and .match() = {len(matches)}")

### Filtraggio dei match

In [ ]:
filtered_matches = filter_matches_by_euclidean_distance(matches, keypoints)
print(f"SIFT matches with CLAHE and .match() + Filter (distanza euclidea) = {len(filtered_matches)}")

### Display dei match

In [ ]:
drawn_matches = cv2.drawMatches(
    label_free, keypoints[0],
    stained, keypoints[1],
    filtered_matches,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    matchesThickness=5
)
print(f"Matches: {len(filtered_matches)}")
show_image(drawn_matches, "Corrispondenze SIFT")

### Allineamento dell'immagine con RANSAC

In [ ]:
points = extract_points(filtered_matches, keypoints)
warp_matrix = compute_ransac_transform(points[0], points[1])
aligned_stained = apply_ransac_transform(stained, warp_matrix, shape=label_free.shape)
aligned_stained_gray = cv2.cvtColor(aligned_stained, cv2.COLOR_BGR2GRAY)
aligned_stained_equalized = cv2.equalizeHist(aligned_stained_gray)
aligned_stained_clahe = apply_clahe(aligned_stained_gray)

difference = cv2.absdiff(label_free_equalized, aligned_stained_equalized)

show_side_images(label_free, "Non colorata", aligned_stained, "Colorata allineata")
show_side_images(label_free_gray, "Non colorata", aligned_stained_gray, "Colorata allineata", cmap='gray')
show_side_images(label_free_equalized, "Non colorata", aligned_stained_equalized, "Colorata allineata", cmap='gray')
show_side_images(label_free_clahe, "Non colorata", aligned_stained_clahe, "Colorata allineata", cmap='gray')
show_image(difference, "Differenza assoluta tra immagini allineate", cmap='gray')

## Allinamento tramite Feature Matching con SIFT e immagini monocromatiche CLAHE

In [ ]:
matches, keypoints = compute_sift_matches(label_free_clahe, stained_clahe)
print(f"SIFT con CLAHE e .match(): {len(matches)}")

### Filtraggio dei match

In [ ]:
filtered_matches = filter_matches_by_euclidean_distance(matches, keypoints)
print(f"SIFT con CLAHE e .match() e filtro euclideo: {len(filtered_matches)}")

### Display dei match

In [ ]:
drawn_matches = cv2.drawMatches(
    label_free, keypoints[0],
    stained, keypoints[1],
    filtered_matches[:20],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    matchesThickness=5
)
print(f"Matches: {len(filtered_matches)}")
show_image(drawn_matches, "Corrispondenze SIFT")

### Allineamento dell'immagine con RANSAC

In [ ]:
points = extract_points(filtered_matches, keypoints)
warp_matrix = compute_ransac_transform(points[0], points[1])
aligned_stained = apply_ransac_transform(stained, warp_matrix, shape=label_free.shape)
aligned_stained_gray = cv2.cvtColor(aligned_stained, cv2.COLOR_BGR2GRAY)
aligned_stained_equalized = cv2.equalizeHist(aligned_stained_gray)
aligned_stained_clahe = apply_clahe(aligned_stained_gray)

difference = cv2.absdiff(label_free_equalized, aligned_stained_equalized)

show_side_images(label_free, "Non colorata", aligned_stained, "Colorata allineata")
show_side_images(label_free_gray, "Non colorata", aligned_stained_gray, "Colorata allineata", cmap='gray')
show_side_images(label_free_equalized, "Non colorata", aligned_stained_equalized, "Colorata allineata", cmap='gray')
show_side_images(label_free_clahe, "Non colorata", aligned_stained_clahe, "Colorata allineata", cmap='gray')
show_image(difference, "Differenza assoluta tra immagini allineate", cmap='gray')